# Compute large embeddings

Computes Parquet embedding datasets for the six local encoders too large to run outside Colab: bge-multilingual-gemma2, Qwen3-Embedding-8B, KaLM-Embedding-Gemma3-12B, Llama-Embed-Nemotron-8B, Harrier-OSS-v1-27B, F2LLM-v2-14B.

Runtime > Change runtime type > pick a GPU runtime with as much VRAM as possible.

| Model | Load dtype | Approx. memory |
| --- | --- | --- |
| bge-multilingual-gemma2 | float16 | ~18.5 GB |
| Qwen3-Embedding-8B | auto (bf16, native) | ~15.1 GB |
| KaLM-Embedding-Gemma3-12B | bfloat16 | ~23.5 GB |
| Llama-Embed-Nemotron-8B | bfloat16 | ~16 GB |
| Harrier-OSS-v1-27B | auto (native dtype) | ~54 GB (estimated, not measured; 27B params, needs >40GB VRAM) |
| F2LLM-v2-14B | bfloat16 | ~28 GB (estimated, not measured) |

A 40GB A100 fits any of bge-multilingual-gemma2, Qwen3-Embedding-8B, KaLM-Embedding-Gemma3-12B, Llama-Embed-Nemotron-8B, or F2LLM-v2-14B plus activations. A 16GB T4 fits only Qwen3-Embedding-8B or Llama-Embed-Nemotron-8B, and even those are tight. Harrier-OSS-v1-27B's estimated ~54GB exceeds a 40GB A100; it needs an 80GB A100/H100 or 8-bit loading.

Llama-Embed-Nemotron-8B is licensed for non-commercial, research use only (NVIDIA's customized-nscl-v1).

In [ ]:
!rm -rf /content/tehillim-representations
!git clone --depth 1 --filter=blob:none --no-checkout \
    https://github.com/rdtaylorjr/tehillim-representations.git /content/tehillim-representations
!git -C /content/tehillim-representations sparse-checkout set --no-cone src pyproject.toml
!git -C /content/tehillim-representations checkout
!cd /content/tehillim-representations && pip install .

Set `MODEL_CHOICE` to `"bge"`, `"qwen3"`, `"kalm"`, `"llama-nemotron"`, `"harrier"`, or `"f2llm"` to run just one model, or leave it `None` to run all six. Set `VARIATION_CHOICE` to `"consonantal"`, `"vocalized"`, or `"cantillation"` to run just one variation, or leave it `None` to run every variation for the chosen model. `generate_local` treats an already-written `.tf` file as done, so this is safe to re-run after a partial failure.

In [ ]:
import subprocess
from pathlib import Path

from semantic.corpus import DEFAULT_BHSA_TF_PATH, Corpus
from semantic.generate import generate_local
from semantic.large_models import ensure_corpus_data, gpu_memory_summary, models_for_choice

MODEL_CHOICE = None
VARIATION_CHOICE = None


def _clone(url: str, destination: Path) -> None:
    subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)


bhsa_path = ensure_corpus_data(
    bhsa_default=DEFAULT_BHSA_TF_PATH,
    data_dir=Path("/content/_bhsa_data"),
    clone=_clone,
)

corpus = Corpus.load(bhsa_path)
psalms = corpus.psalms()
print(f"{len(psalms)} psalms loaded")

output_root = Path("/content/tehillim-representations")
data_dir = output_root / "data" / "type=semantic"
files_before_this_run = set(data_dir.rglob("part-0.parquet")) if data_dir.exists() else set()

for slug, model_name, torch_dtype in models_for_choice(MODEL_CHOICE):
    print(f"computing {model_name} (torch_dtype={torch_dtype})...")
    written = generate_local(
        psalms, output_root, slug, variation=VARIATION_CHOICE, torch_dtype=torch_dtype
    )
    print(f"  wrote {written}")
    summary = gpu_memory_summary()
    if summary:
        print(f"  [GPU memory] {summary}")

print("done")

Download only the Parquet files that appeared under `data/type=semantic/` since this cell started (compared against the directory listing taken before `generate_local` ran, not the full directory, which also holds every file already committed in the repo): the zip preserves the Hive-partitioned directory structure, so unzipping it into the repo root reproduces the correct paths directly, no manual placement needed. Works no matter how many times the cell above was re-run, since it reads real file state, not a record of which call wrote what.

In [ ]:
import zipfile

from google.colab import files

new_files = sorted(set(data_dir.rglob("part-0.parquet")) - files_before_this_run)

if not new_files:
    print("nothing new on disk since this notebook's run cell started")
else:
    with zipfile.ZipFile("/content/data.zip", "w") as zf:
        for path in new_files:
            zf.write(path, arcname=str(path.relative_to(output_root)))
    files.download("/content/data.zip")